In [ ]:
%load_ext autoreload
%autoreload 2

import sys, importlib, shutil
from pathlib import Path

workspace_root = str(Path.cwd().parent)
if workspace_root not in sys.path:
    sys.path.insert(0, workspace_root)
    
shutil.rmtree(Path(workspace_root) / 'utils/__pycache__', ignore_errors=True)

import utils.experiment
importlib.reload(utils.experiment)

from utils.experiment import Experiment
from qiskit_ibm_runtime import QiskitRuntimeService
# # QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token="j7bhgwFwgY0ZiyPYPuGjIk6oFwWK6jivrkRxg9oWxlNs", overwrite=True)
# service = QiskitRuntimeService()
# backend = service.backend("ibm_marrakesh")
# print(f"Using backend: {backend.name}")
backend = None  # Set to None to use the default simulator backend


a = 1  # decay constant
Z = 1  # nuclear charge
center_distance = 1.4
n_qubits = 8
max_range = 32
shots = 1024

exp = Experiment(
    allow_measurement=False,
    optimize_t_gates=False,
    enable_twirling=False,
    enable_measure_mitigation=False,
    enable_zne=False,
    ibm_backend=backend,
    zne_noise_factors=[1, 2, 3],
    zne_extrapolator = 'linear',
)
# experiment_results = exp.run_single_s1_1d_overlap_integral(n_qubits, a, max_range, 1.4, shots)

# experiment_results = exp.run_single_s1_1d_kinetic_analytical(n_qubits, a, max_range, 1.4, shots)

# experiment_results = exp.run_single_s1_1d_kinetic_finite_diff(n_qubits, a, max_range, 1.4, shots)

# experiment_results = exp.run_single_s1_1d_kinetic_integral(n_qubits, a, max_range, 1.4, shots)

experiment_results = exp.run_s2_s1_3d_overlap_integral(n_qubits, a, max_range, shots)


experiment_results.print()




The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


c:\Users\sorin\anaconda3\envs\qiskit21\Lib\site-packages\qiskit_ibm_runtime\fake_provider\local_service.py:274: UserWarning: Options {'dynamical_decoupling': {'enable': True, 'sequence_type': 'XY4'}} have no effect in local testing mode.
  warnings.warn(f"Options {options_copy} have no effect in local testing mode.")



  Center distance (discretized) : None
  Scaled distance (in grid) : None
  Exact result             : 0.000000

  Backend name         : fake_sherbrooke
  Qubits (transpiled)  : 127
  Depth                : 1384
  Single-qubit gates   : 2042  (rz + sx + x)
  Two-qubit gates (ecr): 314
  T gates (logical)    : 0
  All gate counts      : {'rz': 1203, 'sx': 830, 'ecr': 314, 'x': 9}


  Analytical (statevector) result : 0.000006
  Error vs continuous  : 0.000006  (0.00 %)
  Discretisation error : 0.000006  (0.00 %)
  Shot noise           : 0.000000  (0.00 %)

  Sampled (counts) result : 0.000000
  Error vs continuous  : 0.000000  (0.00 %)
  Discretisation error : 0.000006  (0.00 %)
  Shot noise           : 0.000006  (100.00 %)

  Noisy (simulation) result : 0.120038
  Error vs continuous  : 0.120038  (0.00 %)
  Discretisation error : 0.000006  (0.00 %)
  Shot noise           : 0.120033  (2094636.87 %)

  Noisy (estimation) result : 0.125000
  Error vs continuous  : 0.125000  (0.00 %)
  D

# 2p Orbital in Spherical Coordinates — MPS Analysis

## Background: how the 1s and 2s circuits encode the radial integral

In spherical coordinates the 3D inner product of two radial orbitals is

$$\langle A | B \rangle = \int_0^\infty \chi_A^*(r)\,\chi_B(r)\,dr$$

where $\chi(r) = r \cdot R(r)$ is the *reduced* radial function and $R(r)$ is the ordinary radial part.
Including the $r$ factor in the amplitude lets us evaluate the 3D overlap as a plain 1D sum:

$$\langle A | B \rangle \approx \sum_{r=0}^{2^n-1} f_A(r)^*\, f_B(r)$$

The MPS circuits encode $f(r)$ as quantum amplitudes.  The "Jacobian" versions therefore multiply by the extra factor of $r$:

| Orbital | Amplitudes $f(r)$ | Bond dim $\chi$ |
|---|---|---|
| 1s (Jacobian) | $r \cdot e^{-\alpha r}$ | 2 |
| 2s (Jacobian) | $r \cdot (2 - \alpha r)\, e^{-\alpha r/2}$ | 3 |
| **2p (Jacobian)** | $r^2 \cdot e^{-\alpha r/2}$ | **3** |

---

## Transfer matrix structure for 2p

The radial coordinate is represented in binary: $r = \sum_k s_k\, p_k$ with $p_k = 2^{n-1-k}$.

The bond tracks the tuple $(1,\; r_\text{acc},\; r_\text{acc}^2)$ so that after processing all $n$ bits the right boundary can extract $r^2$.

The interior transfer matrices are **identical** to those of the 2s Jacobian:

$$A^{[k]0} = I_3, \qquad A^{[k]1} = e_k \begin{pmatrix} 1 & p_k & p_k^2 \\ 0 & 1 & 2p_k \\ 0 & 0 & 1 \end{pmatrix}, \qquad e_k = e^{-\alpha p_k/2}$$

When bit $s_k = 1$ is added the accumulator updates as:
- $r_\text{new} = r_\text{acc} + p_k$, so $v_1 \to v_1 + p_k$
- $r_\text{new}^2 = r_\text{acc}^2 + 2p_k r_\text{acc} + p_k^2$, so $v_2 \to v_2 + 2p_k v_1 + p_k^2$

This matches the matrix exactly.

### Boundary vectors

The only difference from 2s Jacobian is the **right boundary**, which selects which polynomial of $r$ to extract:

| | Left boundary | Right boundary |
|---|---|---|
| 2s Jacobian | $(1,\; 0,\; 0)$ | $(0,\; 2,\; -\alpha)^T$ — extracts $r(2-\alpha r)$ |
| **2p Jacobian** | $(1,\; 0,\; 0)$ | $(0,\; 0,\; 1)^T$ — extracts $r^2$ |

### Concrete MPS tensors

**k = 0** (left boundary, shape $1\times 2\times 3$):
$$A^{[0]0} = \tfrac{1}{N}(1,\;0,\;0), \qquad A^{[0]1} = \tfrac{e_0}{N}(1,\;p_0,\;p_0^2)$$

**0 < k < n-1** (interior, shape $3\times 2\times 3$):
$$A^{[k]0} = I_3, \qquad A^{[k]1} = e_k\begin{pmatrix}1&p_k&p_k^2\\0&1&2p_k\\0&0&1\end{pmatrix}$$

**k = n-1** (right boundary, shape $3\times 2\times 1$):
$$A^{[n-1]0} = (0,\;0,\;1)^T, \qquad A^{[n-1]1} = (e_k p_k^2,\;2 e_k p_k,\;e_k)^T$$

---

## Norm

$$\|f\|^2 = \sum_{r=0}^{2^n-1} r^4\, e^{-\alpha r}$$

Computed via the same $9\times 9$ transfer matrix product as the 2s Jacobian norm, but with right boundary $(0,0,1)$ instead of $(0,2,-\alpha)$:

```python
l = np.array([1.0, 0.0, 0.0])
r_vec = np.array([0.0, 0.0, 1.0])   # ← only change from 2s norm
v = np.kron(l, l)
for k in range(n):
    p_k = 2.0 ** (n - 1 - k)
    e_k = np.exp(-p_k * alpha / 2.0)
    N_k = e_k * np.array([[1, p_k, p_k**2], [0, 1, 2*p_k], [0, 0, 1]])
    T_k = np.eye(9) + np.kron(N_k, N_k)
    v = v @ T_k
norm_sq = v @ np.kron(r_vec, r_vec)
```

---

## Angular part

The 1s and 2s orbitals have $l=0$, so their spherical harmonic $Y_0^0$ is a constant and the angular integral always equals 1 — the circuit only needs to encode the radial part.

The 2p orbital has $l=1$ with spherical harmonics $Y_1^m$.  Two consequences:

- **Self-overlap** $\langle 2p | 2p \rangle$: the angular integral $\int |Y_1^m|^2\,d\Omega = 1$, so only the radial MPS above is needed.
- **Cross-overlaps** $\langle 1s | 2p \rangle$ and $\langle 2s | 2p \rangle$: the angular integral $\int Y_0^0 (Y_1^m)^*\,d\Omega = 0$ by orthogonality, so these are exactly zero regardless of the radial part — no circuit is needed.
- **Matrix elements with an anisotropic potential** (e.g. $V \propto \cos\theta$): the angular integral no longer factorises to 1 or 0, and you would need a separate angular register encoding $\cos\theta$. For the isotropic Coulomb potential the angular part still factorises and can be computed analytically.